In [1]:
!pip install -q google-generativeai

In [ ]:
import os
import logging
import warnings
import urllib3
import pandas as pd
import time
from tqdm import tqdm  # Để hiển thị progress bar
from datetime import datetime, timedelta

# Suppress all warnings
warnings.filterwarnings('ignore')
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
logging.getLogger().setLevel(logging.ERROR)

import google.generativeai as genai
import requests
from PIL import Image
from io import BytesIO

# Danh sách API keys
API_KEYS = [
    "AIzaSyBnvv2aFHTM...",
    "AIzaSyAnK6Vt-WGd...",
    "AIzaSyAxF533_YAS...",
]

# Quản lý API và rate limit
class APIManager:
    def __init__(self, api_keys):
        self.api_keys = api_keys
        self.current_key_index = 0
        self.request_counts = {key: 0 for key in api_keys}
        self.minute_start_times = {key: datetime.now() for key in api_keys}
        self.day_start_times = {key: datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) for key in api_keys}
        self.rpm_limit = 15  # Requests per minute
        self.rpd_limit = 1500  # Requests per day

    def get_current_api_key(self):
        return self.api_keys[self.current_key_index]

    def switch_api_key(self, error_message=None):
        old_key = self.get_current_api_key()
        self.current_key_index = (self.current_key_index + 1) % len(self.api_keys)
        new_key = self.get_current_api_key()
        
        if error_message:
            print(f"API Key Error: {error_message}")
        
        print(f"Switching from API key {old_key[-5:]} to {new_key[-5:]}")
        init_gemini(new_key)
        return new_key

    def track_request(self):
        key = self.get_current_api_key()
        now = datetime.now()
        
        # Reset minute counter if needed
        if (now - self.minute_start_times[key]).total_seconds() > 60:
            self.request_counts[key] = 0
            self.minute_start_times[key] = now
            
        # Reset day counter if needed
        today_start = now.replace(hour=0, minute=0, second=0, microsecond=0)
        if self.day_start_times[key] < today_start:
            self.day_start_times[key] = today_start
            self.request_counts[key] = 0
            
        # Increment counter
        self.request_counts[key] += 1
        
        # Check if we need to switch keys due to rate limits
        if self.request_counts[key] >= self.rpm_limit:
            return self.switch_api_key(f"Rate limit reached for API key ending with {key[-5:]} ({self.request_counts[key]} requests in the last minute)")
        
        return key
    
    def handle_error(self, error):
        error_str = str(error).lower()
        if "429" in error_str or "quota" in error_str or "exhausted" in error_str:
            return self.switch_api_key(f"Quota exceeded for API key ending with {self.get_current_api_key()[-5:]}")
        return None

def init_gemini(api_key):
    """Initialize Gemini API"""
    genai.configure(api_key=api_key)

def load_image_from_url(url):
    """Load image from URL with resize"""
    try:
        response = requests.get(url, timeout=10, verify=False)  # Bỏ qua SSL verify
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))
        
        # Resize image if too large
        max_size = (800, 800)  # Giới hạn kích thước tối đa
        if image.size[0] > max_size[0] or image.size[1] > max_size[1]:
            image.thumbnail(max_size, Image.Resampling.LANCZOS)
            
        return image
    except Exception as e:
        print(f"Error loading image from URL: {e}")
        return None

def get_prediction(image_url, prompt, api_manager, max_retries=3):
    """Get prediction with retries and API key management"""
    for attempt in range(max_retries):
        try:
            # Track request and potentially switch API key if rate limited
            current_key = api_manager.track_request()
            print(f"\nUsing API key: ...{current_key[-5:]}")
            print(f"Processing image URL: {image_url}")
            
            model = genai.GenerativeModel('gemini-2.0-flash')
            image = load_image_from_url(image_url)
            if image is None:
                print("Failed to load image")
                return None

            print("Generating caption...")
            response = model.generate_content([prompt, image])
            caption = response.text
            print(f"Generated caption: {caption}")
            return caption
            
        except Exception as e:
            error_str = str(e)
            print(f"Attempt {attempt+1}/{max_retries}: {error_str}")
            
            if "429" in error_str or "quota" in error_str or "exhausted" in error_str:
                api_manager.handle_error(e)
                time.sleep(1)
                continue
                
            if attempt == max_retries - 1:
                print(f"Error generating content after {max_retries} attempts: {e}")
                return None
                
            time.sleep(2 * (attempt + 1))

def process_dataset(csv_path, prompt, api_manager, batch_size=10):
    try:
        df = pd.read_csv(csv_path)
        print(f"\nLoaded {len(df)} rows from CSV")
        print(f"Using {len(API_KEYS)} API keys")
        
        # Determine output file name from input path
        file_name = os.path.basename(csv_path)
        output_path = "/kaggle/working/" + file_name
        backup_path = "/kaggle/working/backup_" + file_name
        
        df.to_csv(backup_path, index=False, encoding='utf-8-sig')
        print(f"Created backup at {backup_path}")
        
        # Count completed and remaining items
        completed = df['short_caption'].notna().sum()
        total = len(df)
        print(f"Already completed: {completed}/{total} ({completed/total*100:.2f}%)")
        
        for idx in tqdm(range(len(df))):
            if pd.notna(df.at[idx, 'short_caption']):
                continue
                
            url = df.at[idx, 'original_url']
            print(f"\n--- Processing row {idx+1}/{len(df)} ---")
            caption = get_prediction(url, prompt, api_manager)
            
            if caption:
                df.at[idx, 'short_caption'] = caption
                print(f"Successfully saved caption for row {idx+1}")
                
            if idx % batch_size == 0 and idx > 0:
                df.to_csv(output_path, index=False, encoding='utf-8-sig')
                print(f"\nProgress saved at row {idx}")
                print(f"Completion: {((idx+1)/len(df))*100:.2f}%")
                time.sleep(1)
            
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print("\nProcessing completed! File saved to /kaggle/working/")
        
    except Exception as e:
        print(f"Error processing dataset: {e}")
        # Lưu tiến độ ngay cả khi gặp lỗi
        if 'df' in locals() and 'output_path' in locals():
            df.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"Saved progress before error at {output_path}")
        return None

# Optimized prompt
OPTIMIZED_PROMPT = """
You are a visually impaired person listening to a description of the surrounding traffic situation. Briefly and objectively describe the following criteria in a short paragraph:  
        **Traffic condition**:  
            - Describe the current traffic condition in one simple sentence, focusing on the main vehicle, traffic signs, traffic lights, people, and the scene in the image.  
        **Position of fixed objects**:  
            - Clearly state positions such as "left", "right", "ahead", "center", "on the side" for fixed objects like traffic signs, traffic lights, or police booths.  
            - If there are traffic signs or lights, mention their content clearly.  
        **Lane and movement direction**:  
            - Specify whether vehicles are moving in the same or opposite direction from me (based on the image's perspective).  
            - If a vehicle is crossing, state its direction (e.g., "from left to right", "from right to left").  
        **Viewpoint in the image**:  
            - Determine your position relative to the camera view (e.g., standing on the sidewalk, in the middle of the road, or viewing from a distance). Refer to yourself as "you".  
        **Safe mobility**:  
            - Accurately identify the "left", "right", "ahead", or "center" position of lanes with sidewalks, pedestrian crossings, or lanes without obstacles that affect safe movement.  
        **Include the following constraints**:  
            - Use only simple sentences with clear subjects, verbs, and objects.  
            - Use natural, spoken-like language that is easy to understand.  
            - Do not describe obvious information.  
            - Separate ideas with periods (.) and do not use commas (,) or semicolons (;) to connect sentences.  
            - Do not add emotion or speculation. Do not exceed 20 words. If longer, shorten the sentence.  
            - All sentences must be in a single line without line breaks.
"""

if __name__ == "__main__":
    # Khởi tạo API Manager
    api_manager = APIManager(API_KEYS)
    
    # Initialize Gemini với API key đầu tiên
    init_gemini(api_manager.get_current_api_key())
    
    # Kaggle input dataset path
    # csv_path = "/kaggle/input/csv-v3/standard_dataset_11k.csv"
    csv_path = "/kaggle/input/csv-v3/without_captions.csv"
    process_dataset(csv_path, OPTIMIZED_PROMPT, api_manager)


Loaded 2170 rows from CSV
Using 15 API keys
Created backup at /kaggle/working/backup_without_captions.csv
Already completed: 0/2170 (0.00%)


  0%|          | 0/2170 [00:00<?, ?it/s]


--- Processing row 1/2170 ---

Using API key: ...-tWYI
Processing image URL: https://file1.dangcongsan.vn/data/0/images/2022/12/23/upload_34/viaa-he.jpg


  0%|          | 1/2170 [00:10<6:02:14, 10.02s/it]

Error loading image from URL: HTTPSConnectionPool(host='file1.dangcongsan.vn', port=443): Max retries exceeded with url: /data/0/images/2022/12/23/upload_34/viaa-he.jpg (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7bd07a77a740>, 'Connection to file1.dangcongsan.vn timed out. (connect timeout=10)'))
Failed to load image

--- Processing row 2/2170 ---

Using API key: ...-tWYI
Processing image URL: https://file1.dangcongsan.vn/DATA/0/2017/04/via_he-09_54_55_246.gif


...

Generated caption: Giao thông hỗn loạn có xe tải và ô tô bị tai nạn.  Biển báo và đèn tín hiệu phía trước.  Xe cộ cùng chiều và ngược chiều. Bạn đứng bên lề đường. Vỉa hè phía trước an toàn.

Successfully saved caption for row 2168

--- Processing row 2169/2170 ---

Using API key: ...qO2MQ
Processing image URL: https://images2.thanhnien.vn/528068263637045248/2024/6/10/3-o-to-tai-nan-lien-hoan-vi-khong-giu-khoang-cach-an-toan-1-xe-thanhnien-17180127479891105376027.jpg
Generating caption...


100%|█████████▉| 2169/2170 [2:55:46<00:03,  3.58s/it]

Generated caption: Giao thông đang ùn tắc với nhiều xe ô tô phía trước.  Biển báo và đèn tín hiệu không có.  Vỉa hè nằm bên phải.  Các xe cùng chiều di chuyển phía trước bạn.  Bạn đang ở trong xe, giữa đường.  Vỉa hè bên phải đảm bảo di chuyển an toàn.

Successfully saved caption for row 2169

--- Processing row 2170/2170 ---

Using API key: ...qO2MQ
Processing image URL: https://icdn.dantri.com.vn/OlRTd1upOguwZHvZpzsc/Image/2013/05/1-82e7a.jpg
Generating caption...


100%|██████████| 2170/2170 [2:55:50<00:00,  4.86s/it]

Generated caption: Một xe tải hư hỏng nặng nằm chính giữa ảnh. Không có biển báo hoặc đèn tín hiệu.  Xe tải nằm bên lề đường.  Không có phương tiện khác. Bạn đứng trên vỉa hè.  Vỉa hè nằm bên trái. Đường đi an toàn ở bên trái.

Successfully saved caption for row 2170

Processing completed! File saved to /kaggle/working/
